# Notebook 01: Dataset Preparation
**Thesis Phase:** Ingesting GSM8K and GSM-Symbolic  
**Outputs:** 
- `data/raw/gsm8k_train_5k.json` (5,000 distillation sources)
- `data/raw/gsm_symbolic_test.json` (Evaluation benchmark)


In [2]:
import os
import json
from datasets import load_dataset

DATA_RAW_DIR = "../data/raw"
os.makedirs(DATA_RAW_DIR, exist_ok=True)
print(f"Target raw directory: {os.path.abspath(DATA_RAW_DIR)}")


Target raw directory: /home/starf/dual-lora-faithfulness/data/raw


### 1. Load GSM8K (5,000 Problem Subset)


In [3]:
print("Loading GSM8K (openai/gsm8k)...")
gsm8k_dataset = load_dataset("gsm8k", "main", split="train")

# Select 5,000 problems as specified in thesis
gsm8k_5k = gsm8k_dataset.select(range(min(5000, len(gsm8k_dataset))))

gsm8k_data_list = []
for idx, sample in enumerate(gsm8k_5k):
    question = sample["question"]
    raw_answer = sample["answer"]
    final_numeric = raw_answer.split("####")[-1].strip() if "####" in raw_answer else ""
    
    gsm8k_data_list.append({
        "id": idx,
        "question": question,
        "ground_truth_cot": raw_answer,
        "target_answer": final_numeric
    })

gsm8k_out_path = os.path.join(DATA_RAW_DIR, "gsm8k_train_5k.json")
with open(gsm8k_out_path, "w", encoding="utf-8") as f:
    json.dump(gsm8k_data_list, f, indent=2)
print(f"✓ Saved 5,000 GSM8K problems to: {gsm8k_out_path}")


Loading GSM8K (openai/gsm8k)...
✓ Saved 5,000 GSM8K problems to: ../data/raw/gsm8k_train_5k.json


### 2. Load GSM-Symbolic (Apple Benchmark)


In [4]:
print("Loading GSM-Symbolic (apple/GSM-Symbolic)...")
try:
    gsm_symbolic_ds = load_dataset("apple/GSM-Symbolic", name="main", split="test")
    gsm_symbolic_list = []
    
    for sample in gsm_symbolic_ds:
        question = sample["question"]
        raw_answer = sample["answer"]
        final_numeric = raw_answer.split("####")[-1].strip() if "####" in raw_answer else ""
        
        gsm_symbolic_list.append({
            "id": sample.get("id", len(gsm_symbolic_list)),
            "instance": sample.get("instance", 0),
            "question": question,
            "ground_truth_cot": raw_answer,
            "target_answer": final_numeric
        })

    symbolic_out_path = os.path.join(DATA_RAW_DIR, "gsm_symbolic_test.json")
    with open(symbolic_out_path, "w", encoding="utf-8") as f:
        json.dump(gsm_symbolic_list, f, indent=2)
    print(f"✓ Saved {len(gsm_symbolic_list)} GSM-Symbolic evaluation samples to: {symbolic_out_path}")
except Exception as e:
    print(f"⚠️ Could not load apple/GSM-Symbolic directly: {e}")


Loading GSM-Symbolic (apple/GSM-Symbolic)...
✓ Saved 5000 GSM-Symbolic evaluation samples to: ../data/raw/gsm_symbolic_test.json


### 3. Inspect Ingested Sample


In [5]:
sample_item = gsm8k_data_list[0]
print(f"QUESTION:\n{sample_item['question']}\n")
print(f"GROUND TRUTH ANSWER:\n{sample_item['ground_truth_cot']}\n")
print(f"EXTRACTED NUMERIC TARGET: {sample_item['target_answer']}")


QUESTION:
Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

GROUND TRUTH ANSWER:
Natalia sold 48/2 = <<48/2=24>>24 clips in May.
Natalia sold 48+24 = <<48+24=72>>72 clips altogether in April and May.
#### 72

EXTRACTED NUMERIC TARGET: 72
